# Checklist baseline

The high-risk checklist every model must beat (flags in `../../../contracts/cohort_config.yaml`):
age < 18 or 35+, prior_live_births >= 5, preceding_interval_m < 24, multiple, prior_stillbirth.

Build the flag, get its flag rate, and count how many perinatal deaths it catches.
That flag rate is what the model is compared against.

All three cohorts, on the same data and split as the models: `build_cohort()` and its 80/20
cluster split from `ml/src/dataset.py`. Training rows are shown for reference; the test rows are
the comparison the model notebooks use (flag the same share of test rows, count events caught).

## 1 — Load the cohorts

In [ ]:
import sys
from pathlib import Path

# make the repo root importable when run from ml/notebooks/modeling
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml" / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score
from ml.src.dataset import COHORTS, OUTCOMES, build_cohort

# combined, cleaned cohorts (same rows and split as every model notebook)
data = {c: build_cohort(c) for c in COHORTS}
for c, df in data.items():
    print(f"{c:14s} rows {len(df):6,}  train {int((df['split'] == 'train').sum()):6,}  "
          f"test {int((df['split'] == 'test').sum()):5,}")

## 2 — The checklist flag
Same rule as `checklist_flag` in `dataset.py` (not in `SELECTED`, so rebuilt here from the kept
columns). Imputed intervals count as their imputed value; first pregnancies have no interval and
never meet the interval item. Two scores are scored:

- **flag**: any item (0/1). This is the checklist as used in clinics.
- **items**: number of items met (0–5). A ranking version of the checklist, for a fairer PR-AUC.

In [ ]:
def checklist_items(d: pd.DataFrame) -> pd.DataFrame:
    """One 0/1 column per checklist item."""
    return pd.DataFrame({
        "age_lt18_or_35plus": (d["maternal_age"] < 18) | (d["maternal_age"] >= 35),
        "parity_5plus": d["prior_live_births"] >= 5,
        "interval_lt24": d["preceding_interval_m"] < 24,
        "multiple": d["multiple"] == 1,
        "prior_stillbirth": d["prior_stillbirth"] == 1,
    }).astype(int)


items = {c: checklist_items(df) for c, df in data.items()}

# share of test rows meeting each item
item_share = pd.DataFrame({c: items[c][data[c]["split"].eq("test").to_numpy()].mean()
                           for c in COHORTS})
print(item_share.round(3))

## 3 — Flag rate, events caught, PR-AUC and ROC-AUC

In [ ]:
def summarise(y: np.ndarray, it: pd.DataFrame) -> dict:
    """Checklist results for one set of rows."""
    flag = it.any(axis=1).to_numpy()
    n_items = it.sum(axis=1).to_numpy()
    caught = int((flag & (y == 1)).sum())
    return {
        "rows": len(y), "events": int(y.sum()), "event_rate": y.mean(),
        "flagged": int(flag.sum()), "flag_rate": flag.mean(), "events_caught": caught,
        "sensitivity": caught / y.sum(), "ppv": caught / flag.sum(),
        "pr_auc_flag": average_precision_score(y, flag), "roc_auc_flag": roc_auc_score(y, flag),
        "pr_auc_items": average_precision_score(y, n_items),
        "roc_auc_items": roc_auc_score(y, n_items),
    }


rows = {}
for c, df in data.items():
    y = df[OUTCOMES[c]].to_numpy()
    for split in ["train", "test"]:
        for country in ["pooled", "Kenya", "Tanzania"]:
            m = df["split"].eq(split).to_numpy()
            if country != "pooled":
                m &= df["country"].eq(country).to_numpy()
            rows[(c, split, country)] = summarise(y[m], items[c][m])

results = pd.DataFrame(rows).T
results.index.names = ["cohort", "split", "country"]
results = results.astype({k: int for k in ["rows", "events", "flagged", "events_caught"]})
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
print(results.round(3))

### Results (test split, pooled)

| cohort | test rows | events | flag rate | flagged | caught | sensitivity | PPV | PR-AUC (flag) | ROC-AUC (flag) | PR-AUC (items) | constant baseline |
|---|---|---|---|---|---|---|---|---|---|---|---|
| perinatal | 6,189 | 220 | 45.4% | 2,808 | 115 | 0.523 | 0.041 | 0.038 | 0.536 | 0.044 | 0.036 |
| perinatal_anc | 3,310 | 108 | 44.0% | 1,457 | 49 | 0.454 | 0.034 | 0.033 | 0.507 | 0.042 | 0.033 |
| lbw | 1,849 | 130 | 41.6% | 770 | 69 | 0.531 | 0.090 | 0.081 | 0.561 | 0.085 | 0.070 |

- The checklist flags over 40% of births in every cohort, yet catches only about half the events.
- It barely beats flagging at random: PPV 0.034–0.090 against event rates of 0.033–0.070, and
  ROC-AUC 0.51–0.56. For perinatal_anc on test it is no better than chance (ROC-AUC 0.507).
- Counting items ranks slightly better than the yes/no flag (PR-AUC 0.042 vs 0.033 for ANC), but
  still close to the constant baseline.
- Test is a little weaker than train for both perinatal cohorts (e.g. ANC sensitivity 0.45 vs
  0.55); the checklist has no fitted parameters, so this is sampling variation between clusters.
- These test figures match the checklist rows in the gradient boosting notebooks (ANC 49 of 108,
  LBW 69 of 130, perinatal 115 of 220), so every model is compared against the same baseline.

In [ ]:
# test sensitivity and PPV by cohort, against the event rate (PPV of random flagging)
test_pooled = results.xs(("test", "pooled"), level=("split", "country"))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
test_pooled["sensitivity"].plot.bar(ax=axes[0], rot=0)
axes[0].axhline(0.5, color="k", ls=":", lw=1)
axes[0].set(title="Checklist sensitivity (test)", ylabel="events caught / events", ylim=(0, 1))
x = np.arange(len(test_pooled))
axes[1].bar(x - 0.2, test_pooled["ppv"], 0.4, label="checklist PPV")
axes[1].bar(x + 0.2, test_pooled["event_rate"], 0.4, label="event rate (random flag)")
axes[1].set_xticks(x, test_pooled.index)
axes[1].set(title="Checklist PPV vs random (test)", ylabel="share of flagged with event")
axes[1].legend()
fig.tight_layout()
plt.show()

## 4 — Save the result

One `checklist` row per cohort in `model_metrics`, so the dashboard has the baseline next to the
models. `recall_at_flag` is the checklist's own sensitivity at its flag rate; `beats_checklist`
is False (it is the checklist). PR-AUC and ROC-AUC are those of the yes/no flag.
To save, set `SAVE_TO_DB = True` below once the app Postgres is up; re-running replaces the rows.

In [ ]:
# rows for model_metrics, from the pooled test results
save_rows = [{
    "cohort": c,
    "algorithm": "checklist",
    "pr_auc": float(test_pooled.loc[c, "pr_auc_flag"]),
    "roc_auc": float(test_pooled.loc[c, "roc_auc_flag"]),
    "recall_at_flag": float(test_pooled.loc[c, "sensitivity"]),
    "beats_checklist": False,
    "test_events": int(test_pooled.loc[c, "events"]),
    "is_best": False,
} for c in COHORTS]
print(pd.DataFrame(save_rows).set_index("cohort").round(3))

SAVE_TO_DB = False   # needs the app DB (docker compose up -d app_db)
if SAVE_TO_DB:
    from db.metrics import save_metrics
    for row in save_rows:
        save_metrics(**row)